# Project 1. Denoising Network

**Student(s):**
- Dawod Ghifari 520140154
- Hilal Chamtie 540749283
- Darin Li 500048292
- Akshar Hossain 540823996

In this project, you're going to implement a neural network to denoise images, there are several parts you need to implement to make the whole pipeline complete.

1. Dataset
2. Metrics
3. Networks
4. Training
5. Additional Question

## Dataset

In this project we are going to use an image dataset of 400 grayscale 180*180 images as our dataset, use command below to download the dataset

In [ ]:
import os
if not os.path.exists('ImageSet'):
    !wget "https://www.dropbox.com/scl/fi/bhvpke5p7u1xolerayx13/ImageSet-1.zip?rlkey=1oz9g3cpomt0wln0es8j9wje6&st=1fj2ksrj&dl=1" -O ImageSet.zip
    !unzip -q ImageSet.zip
else:
    print("ImageSet already exists, skipping download.")

If above link does not work, please use:
https://www.dropbox.com/scl/fi/bhvpke5p7u1xolerayx13/ImageSet-1.zip?rlkey=1oz9g3cpomt0wln0es8j9wje6&st=1fj2ksrj&dl=0
Now you should have a folder called ImageSet, and there're 400 images in it

In [ ]:
!ls ImageSet | wc -l

Now you need to implement two classes, TrainingSet and TestingSet, you should first split your dataset into 350 images and 50 images. TrainingSet should use the 350 images to form a dataset, with each entry being a pair of image tensors, and the first image should be a noisy version of the second original image. In other words, `training_set[i]` should return `[noisy_image(=original_image + noise), original_image]`, and images should be tensors of shape $C\times H\times W$, in this case, $1\times 180\times 180$

TestingSet is the same thing with the remaining 50 images.
1. Please refer to the following code to add noise
    ```python
    def add_noise(img):
        noise = torch.randn(img.size()).mul_(self.sigma/255.0)
        noisy = img + noise
        noisy[torch.where(noisy > 1)] = 1
        noisy[torch.where(noisy < 0)] = 0
        return noisy
    ```
2. Also refer to the following code as how to read images from file to memory
    ```python
    image_path = os.path.join(ROOT_PATH, IMAGE_PATH)
    img = PIL.Image.open(image_path)
    ```

In [ ]:
import os
import torch
import PIL.Image
import numpy as np
from torch.utils.data import Dataset

ROOT_PATH = 'ImageSet'
SIGMA = 10

class ImageDenoiseDataset(Dataset):
    def __init__(self, image_files, sigma=SIGMA, augment=False):
        self.sigma = sigma
        self.augment = augment
        self.images = []
        for fname in image_files:
            image_path = os.path.join(ROOT_PATH, fname)
            img = PIL.Image.open(image_path).convert('L')
            img_tensor = torch.from_numpy(np.array(img, dtype=np.float32) / 255.0).unsqueeze(0)  # 1xHxW
            self.images.append(img_tensor)

    def add_noise(self, img):
        noise = torch.randn(img.size()).mul_(self.sigma / 255.0)
        noisy = img + noise
        noisy[torch.where(noisy > 1)] = 1
        noisy[torch.where(noisy < 0)] = 0
        return noisy

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        original = self.images[idx]
        if self.augment:
            if torch.rand(1) > 0.5:
                original = original.flip(-1)  # horizontal flip
            if torch.rand(1) > 0.5:
                original = original.flip(-2)  # vertical flip
        noisy = self.add_noise(original)
        return [noisy, original]

# Sort filenames for reproducible split
all_files = sorted(os.listdir(ROOT_PATH))
train_files = all_files[:350]
test_files = all_files[350:]

class TrainingSet(ImageDenoiseDataset):
    def __init__(self):
        super().__init__(train_files, augment=True)

class TestingSet(ImageDenoiseDataset):
    def __init__(self):
        super().__init__(test_files, augment=False)
        torch.manual_seed(42)
        self.noisy_images = [self.add_noise(img) for img in self.images]

    def __getitem__(self, idx):
        return [self.noisy_images[idx], self.images[idx]]

You can use the following code block to check if your implementation is correct, first, there should be **no error**, second, the shape of image should be **`[1, 180, 180]`**, and finally, in the drawing area, the **left hand side image should be noisier than the right hand side image**, but they should be images of the same thing.

In [ ]:
training_set = TrainingSet()
testing_set = TestingSet()
assert len(training_set) == 350
assert len(testing_set) == 50

print(f'Shape of image: {training_set[0][0].shape}')

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,2)
axes[0].imshow(training_set[0][0][0], cmap='gray')
axes[0].axis('off')
axes[0].set_title('noisy example')
axes[1].imshow(training_set[0][1][0], cmap='gray')
axes[1].axis('off')
axes[1].set_title('original example')

## Metrics
To quantify how noisy an image is compared to the original one, we're going to use PSNR, please implement a function `psnr` to return the psnr score.

Refer to https://en.wikipedia.org/wiki/Peak_signal-to-noise_ratio about the formula of PSNR

Note:
1. higher PSNR means noise is relatively smaller, the PSNR of the original image is positive infinity, because the noise is zero.
2. the psnr is a symetric function, meaning the psnr of a noisy image with respect to the original one is the same as the psnr of the original image with respect to the noisy one.

In [ ]:
def psnr(original, noisy):
    mse = torch.mean((original - noisy) ** 2)
    if mse == 0:
        return float('inf')
    max_val = 1.0  # images are normalized to [0, 1]
    return 10 * torch.log10(max_val ** 2 / mse)

Run the following code to check if the implementation is correct, the expected output should be about 7.96

In [ ]:
import torch
test_original = torch.tensor([[0.1, 0.2], [0.3, 0.4]])
test_noisy = torch.tensor([[0.5, 0.6], [0.7, 0.8]])
print(f'PSNR score: {psnr(test_original, test_noisy)}')

And we can calculate the psnr score for the noisy image pair we showed above, the score should be aroud 28, but there could be exception.

In [ ]:
sample = training_set[0]
print(f'PSNR score for example images: {psnr(sample[1], sample[0])}')

## Network
Now that we got dataset ready and metrics ready, we start preparing the network. You need to define a class `DenoiseNetwork` as your network class.

The goal of your network is to take the noisy image as input and output the predicted **noise**. First of all, the input and the output of the network should have the same size, the main idea is to predict the original image first by going through several CNN layers, and then use the input noisy image to deduct predicted original image to get the noise, the pseudo code should be like:
```python
class DenoiseNetwork(nn.Module):
    def __init__(self):
        define some cnn layers and other necessary components
    
    def forward(self, x):
        predicted_original_image = cnn_network(x)
        noise = x - predicted_original_image
        return noise
```
Then calculate the mean squared error between the predicted noise and the truth noise as our loss, and try to minimize it.

Tips:
1. you can use nn.MSELoss as your loss function
2. Use Adam instead of SGD as your optimizer, initial learning rate set to 0.001
3. Use `torch.nn.init.orthogonal_` to initialize the `weight` of your cnn layers as orthogonal matrices, and use `torch.nn.init.constant_` to fill the `bias` of your cnn layers with `0`s.
4. Try dropout, batchnorm etc. to improve the results (training speed, restored results etc.)

In [ ]:
import torch.nn as nn
import torch.nn.init as init

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

class DenoiseNetwork(nn.Module):
    def __init__(self, num_channels=64, num_layers=17):
        super(DenoiseNetwork, self).__init__()
        layers = []
        # First layer: Conv + ReLU
        layers.append(nn.Conv2d(1, num_channels, kernel_size=3, padding=1, bias=True))
        layers.append(nn.ReLU(inplace=True))
        # Middle layers: Conv + BN + ReLU
        for _ in range(num_layers - 2):
            layers.append(nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1, bias=False))
            layers.append(nn.BatchNorm2d(num_channels))
            layers.append(nn.ReLU(inplace=True))
        # Last layer: Conv (output 1 channel, predicted clean image)
        layers.append(nn.Conv2d(num_channels, 1, kernel_size=3, padding=1, bias=True))
        self.cnn = nn.Sequential(*layers)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                init.orthogonal_(m.weight)
                if m.bias is not None:
                    init.constant_(m.bias, 0)

    def forward(self, x):
        predicted_clean = self.cnn(x)
        noise = x - predicted_clean
        return noise

    @property
    def device(self):
        return next(self.parameters()).device

net = DenoiseNetwork().to(device)

Here're some basic tests to see if your network can at least run through an example image, this is expected to produce no error.

In [ ]:
example_batch = training_set[0][0].unsqueeze(0)
assert net(example_batch.to(device)).shape == example_batch.shape

Now we need a quantitative score to indicate how well a network performs. Previously we have defined the psnr function, but it only calculates psnr of an image pair, we need to calculate two scores to see how well the network denoises, the first is the mean psnr score of all noisy images, which indicates how noisy these unprocessed images are, and then assume we have the network ready, we can use the network to predict the noise, and deduct the noise from the noisy images to produce restored images, then we calculate the mean psnr score of these restored images with respect to the original images, and this score indicate how noisy the restored images are. If everything works out fine, we should be able to observe a higher psnr on the restored images.

You need to define a `mean_psnr` function that takes a dataset and a network as input and calculate the mean psnr scores of original noisy images across the whole dataset and mean psnr of restored images processed by the network.

In [ ]:
def mean_psnr(testset, net):
    psnr_original_list = []
    psnr_restored_list = []
    net.eval()
    with torch.no_grad():
        for i in range(len(testset)):
            noisy, original = testset[i]
            # PSNR of noisy vs original
            psnr_original_list.append(psnr(original, noisy))
            # Restore using network
            noisy_gpu = noisy.unsqueeze(0).to(net.device)
            predicted_noise = net(noisy_gpu).squeeze(0).cpu()
            restored = (noisy - predicted_noise).clamp(0, 1)
            psnr_restored_list.append(psnr(original, restored))
    net.train()
    mean_psnr_original = torch.stack(psnr_original_list).mean()
    mean_psnr_after = torch.stack(psnr_restored_list).mean()
    return mean_psnr_original, mean_psnr_after

We can calculte the mean psnr on `testing_set`

In [ ]:
mean_psnr(testing_set, net)

If your code is correct, you should see the mean psnr of original images should be around 28, and the psnr of network processed images is much smaller, which means, a randomly initialzed network adds even more noise, you should see this by displaying.

In [ ]:
noisy_image, original_image = testing_set[0]
noisy_image = noisy_image.to(device)
predicted_noise = net(noisy_image.unsqueeze(0)).squeeze(0)
restored_image = (noisy_image - predicted_noise).clamp(0, 1)

fig, axes = plt.subplots(1,3)
fig.set_figwidth(15)
axes[0].imshow(noisy_image[0].cpu(), cmap='gray')
axes[0].axis('off')
axes[0].set_title('noisy')
axes[1].imshow(original_image[0], cmap='gray')
axes[1].axis('off')
axes[1].set_title('original')
axes[2].imshow(restored_image[0].cpu().detach(), cmap='gray')
axes[2].axis('off')
axes[2].set_title('restored')

## Training
Now that we got everything ready, we should start training, in the next section, you need to implement the training process, that includes defining criteria, setting up optimizer, going through several epochs to train the network, during the training, you should also analyze the psnr scores to see how it goes in terms of quantified performance.

Checklist:
1. define dataloader, recommend batch size starting from 32
2. criteria
3. optimizer
4. (optional) consider using functions in torch.optim.lr_scheduler to adjust your learning rate, because smaller learning rate might work better in the later period of training, similar to fine adjustment. Reference: https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
5. during each iteration, you need to 1. get the noisy image and the original image 2. calculate predicted noise from network, use MSE to calculate the distance between predicted noise and true noise 3. reset gradients to zero 3. use the distance as loss to backward the network to get gradients 4. perform learning with the gradients using optimizer
6. From time to time (e.g. each epoch), calculate PSNR on testing_set

In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim

training_set = TrainingSet()
testing_set = TestingSet()

train_loader = DataLoader(training_set, batch_size=32, shuffle=True, num_workers=0)

criterion = nn.MSELoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

num_epochs = 150
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

# Track metrics for plotting
epoch_losses = []
psnr_noisy_history = []
psnr_restored_history = []
eval_epochs = []

for epoch in range(num_epochs):
    net.train()
    epoch_loss = 0.0
    for batch_idx, (noisy, original) in enumerate(train_loader):
        noisy = noisy.to(device)
        original = original.to(device)
        true_noise = noisy - original

        optimizer.zero_grad()
        predicted_noise = net(noisy)
        loss = criterion(predicted_noise, true_noise)
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()

    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    epoch_losses.append(avg_loss)

    # Evaluate every 10 epochs
    if (epoch + 1) % 10 == 0 or epoch == 0:
        psnr_before, psnr_after = mean_psnr(testing_set, net)
        eval_epochs.append(epoch + 1)
        psnr_noisy_history.append(psnr_before.item())
        psnr_restored_history.append(psnr_after.item())
        print(f'Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.6f} | '
              f'PSNR noisy: {psnr_before:.2f} dB | PSNR restored: {psnr_after:.2f} dB | '
              f'LR: {scheduler.get_last_lr()[0]:.6f}')
    else:
        print(f'Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.6f} | LR: {scheduler.get_last_lr()[0]:.6f}')

# Plot training loss and PSNR curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, num_epochs + 1), epoch_losses, linewidth=0.8)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(eval_epochs, psnr_noisy_history, 'o--', label='Noisy PSNR', markersize=4)
ax2.plot(eval_epochs, psnr_restored_history, 's-', label='Restored PSNR', markersize=4)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('PSNR (dB)')
ax2.set_title('PSNR on Test Set')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Now that your net is ready, we can re do the demonstration.

In [ ]:
noisy_image, original_image = testing_set[0]
noisy_image = noisy_image.to(device)
predicted_noise = net(noisy_image.unsqueeze(0)).squeeze(0)
restored_image = (noisy_image - predicted_noise).clamp(0, 1)

fig, axes = plt.subplots(1,3)
fig.set_figwidth(15)
axes[0].imshow(noisy_image[0].cpu(), cmap='gray')
axes[0].axis('off')
axes[0].set_title('noisy')
axes[1].imshow(original_image[0], cmap='gray')
axes[1].axis('off')
axes[1].set_title('original')
axes[2].imshow(restored_image[0].cpu().detach(), cmap='gray')
axes[2].axis('off')
axes[2].set_title('restored')

The network I trained here is a simple 3-layer low number of channel cnn network, and you can see it's already starting to work. Now try adjust some parameters/network structure to make it work even better. Write down your analysis to make a pdf report.

You need to submit two files, this ipynb file and a pdf report with your analysis.

# Additional Question

In this additional question, you need to test on noise different from your training set. In the previous project, you were asked to use Gaussian noise for training, and you also used Gaussian noise when testing.

In this additional question, you are asked to test your trained model using speckle noise. A simple model of Speckle noise is multiplicative noise, which means that the noise is generated by multiplying each pixel value of the image by a random number. This random number is usually drawn from a distribution with a mean of 1 to ensure that noise averaging does not brighten or darken the image.

Below is a Python function that simply implements Speckle noise. This implementation first generates a random noise matrix of the same size as the input image, with values drawn from a normal distribution with mean 1 and standard deviation sigma. This noise matrix is then multiplied by the original image to generate an image with Speckle noise. Finally, the result is constrained to the range 0 to 1 to keep the pixel values valid.

```python
def add_speckle_noise(img, sigma=0.1):
    noise = torch.randn(img.size()) * sigma + 1.0
    noisy_img = img * noise
    noisy_img.clamp_(0, 1)
    return noisy_img
```
You are asked to do the following:

1. First, you are asked to find appropriate sigma values so that the resulting image is visually similar to your training noise. To make your results more general, it is best to choose 3 possible values. (5%)

2. Test your model on these speckle noises using the same model trained above. Compare the PSNR tested on speckle noise with the PSNR tested on Gaussian noise. Show your results in a table. (5%)

3. Finally, draw your conclusion in the pdf report. Can a model trained on Gaussian noise handle speckle noise? To what extent can it be handled? (5%)


In [ ]:
import pandas as pd

def add_speckle_noise(img, sigma=0.1):
    noise = torch.randn(img.size()) * sigma + 1.0
    noisy_img = img * noise
    noisy_img.clamp_(0, 1)
    return noisy_img

class SpeckleTestingSet(Dataset):
    """Testing set with speckle noise instead of Gaussian noise."""
    def __init__(self, sigma_speckle=0.1):
        self.sigma_speckle = sigma_speckle
        self.images = []
        for fname in test_files:
            image_path = os.path.join(ROOT_PATH, fname)
            img = PIL.Image.open(image_path).convert('L')
            img_tensor = torch.from_numpy(np.array(img, dtype=np.float32) / 255.0).unsqueeze(0)
            self.images.append(img_tensor)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        original = self.images[idx]
        noisy = add_speckle_noise(original, self.sigma_speckle)
        return [noisy, original]

# Part 1: Find 3 sigma values for speckle noise that are visually similar to Gaussian noise
speckle_sigmas = [0.04, 0.06, 0.10]

# Visualize speckle noise at different sigma values alongside Gaussian noise
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
sample_img = testing_set.images[0]

axes[0].imshow(sample_img[0], cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

gaussian_noisy = testing_set.add_noise(sample_img)
axes[1].imshow(gaussian_noisy[0], cmap='gray')
axes[1].set_title(f'Gaussian (σ={SIGMA})')
axes[1].axis('off')

for i, s in enumerate(speckle_sigmas):
    speckle_noisy = add_speckle_noise(sample_img, sigma=s)
    axes[i + 2].imshow(speckle_noisy[0], cmap='gray')
    axes[i + 2].set_title(f'Speckle (σ={s})')
    axes[i + 2].axis('off')

plt.tight_layout()
plt.show()

# Part 2: Test model on speckle noise and compare with Gaussian noise
psnr_gauss_before, psnr_gauss_after = mean_psnr(testing_set, net)

results = {
    'Noise Type': [f'Gaussian (σ={SIGMA})'],
    'PSNR Before (dB)': [f'{psnr_gauss_before:.2f}'],
    'PSNR After (dB)': [f'{psnr_gauss_after:.2f}'],
    'Improvement (dB)': [f'{psnr_gauss_after - psnr_gauss_before:.2f}']
}

for s in speckle_sigmas:
    speckle_set = SpeckleTestingSet(sigma_speckle=s)
    psnr_before, psnr_after = mean_psnr(speckle_set, net)
    results['Noise Type'].append(f'Speckle (σ={s})')
    results['PSNR Before (dB)'].append(f'{psnr_before:.2f}')
    results['PSNR After (dB)'].append(f'{psnr_after:.2f}')
    results['Improvement (dB)'].append(f'{psnr_after - psnr_before:.2f}')

df = pd.DataFrame(results)
print("PSNR Comparison: Gaussian vs Speckle Noise")
print(df.to_string(index=False))

# Part 3: Visual comparison of restored images
fig, axes = plt.subplots(len(speckle_sigmas) + 1, 3, figsize=(15, 5 * (len(speckle_sigmas) + 1)))

net.eval()
with torch.no_grad():
    noisy_g, orig_g = testing_set[0]
    pred_noise_g = net(noisy_g.unsqueeze(0).to(device)).squeeze(0).cpu()
    restored_g = (noisy_g - pred_noise_g).clamp(0, 1)

    axes[0, 0].imshow(noisy_g[0], cmap='gray')
    axes[0, 0].set_title(f'Noisy (Gaussian σ={SIGMA})')
    axes[0, 0].axis('off')
    axes[0, 1].imshow(orig_g[0], cmap='gray')
    axes[0, 1].set_title('Original')
    axes[0, 1].axis('off')
    axes[0, 2].imshow(restored_g[0], cmap='gray')
    axes[0, 2].set_title(f'Restored (PSNR={psnr(orig_g, restored_g):.2f})')
    axes[0, 2].axis('off')

    for i, s in enumerate(speckle_sigmas):
        speckle_set = SpeckleTestingSet(sigma_speckle=s)
        noisy_s, orig_s = speckle_set[0]
        pred_noise_s = net(noisy_s.unsqueeze(0).to(device)).squeeze(0).cpu()
        restored_s = (noisy_s - pred_noise_s).clamp(0, 1)

        axes[i+1, 0].imshow(noisy_s[0], cmap='gray')
        axes[i+1, 0].set_title(f'Noisy (Speckle σ={s})')
        axes[i+1, 0].axis('off')
        axes[i+1, 1].imshow(orig_s[0], cmap='gray')
        axes[i+1, 1].set_title('Original')
        axes[i+1, 1].axis('off')
        axes[i+1, 2].imshow(restored_s[0], cmap='gray')
        axes[i+1, 2].set_title(f'Restored (PSNR={psnr(orig_s, restored_s):.2f})')
        axes[i+1, 2].axis('off')

plt.tight_layout()
plt.show()

### Conclusion

A model trained on Gaussian noise can partially handle speckle noise, especially when the speckle noise level produces similar visual degradation. However, since speckle noise is multiplicative (signal-dependent) while Gaussian noise is additive (signal-independent), the model's effectiveness degrades as speckle sigma increases. The model tends to perform best on speckle noise levels that produce similar PSNR to the Gaussian training noise, but the improvement is generally smaller.

# Marking Scheme:


*   Code implementation: 35%


> * Dataset 5% (Download the dataset correctly and preprocess it appropriately for use as the training and testing sets.)
> * Metrics 5% (Set and use appropriate metrics to evaluate the model.)
> * Network 5% (only 5% because network overlaps with results, you need to adjust the network to improve the results anyway.)
> * Training code 10% (The training code executes correctly, and the code structure is clear and well organized.)
> * reasonably good results 10% (The model reaches the target training outcome with satisfactory performance.)


*   PDF report: 25%

> * Basic results demonstration (network introduction, denoising results showcase) 10%
> * Analysis and improvements 15% (You're supposed to clarify how do you make the network work, e.g. if you encounter some issues, what do you do to address them)


*   Addtional question: 15%

> * Find appropriate sigma values (5%)
> * Test your model on these speckle noises and report them in a table (5%)
> * Draw your conclusion: Can a model trained on Gaussian noise handle speckle noise? To what extent can it be handled? (5%)


*   Oral question: 25%

> * After the assignment submission, each group will be asked one question related to the assignment during the tutorial. This is a group-based activity, and each group may nominate one member to answer the question. All group members must attend the session.

